# OpenAlex

As you could see in the previous notebook ``01_uniformize``, the SYNERGY datasets do not contain titles and abstracts.

Within this notebook we therefore retrieve the title and abstract of each respective article by quering the OpenAlex API with the article's OpenAlex identifier.

> **NOTE**
>
> The datasets after running this notebook already exist within the ``data/datasets/02_openalex`` directory.
> If you simply want to inspect the data, you can directly access them there.
>
> When running the code for comprehensiveness, keep in mind that the downloads might take a while, depending on your bandwidth.

## Custom Modules
The ``src`` directory houses custom modules with functions that will be reused throughout the project.

To be able to import these moduls, we begin with programmatically adding the project's root directory to ``sys.path``.

After adding the root to ``sys.path``, we can import the ``data`` and ``util`` modules:

In [ ]:
import os, sys

# recursively search for the root directory containing a specific file
def find_root_dir(search_for='.gitignore'):

    current_dir = os.getcwd()

    while True:
        if os.path.exists(os.path.join(current_dir, search_for)):
            return current_dir
        parent_dir = os.path.dirname(current_dir)
        if parent_dir == current_dir:
            raise FileNotFoundError(f"Could not find '{search_for}' in any parent directory.")
        current_dir = parent_dir


# save the root directory to a variable
root_dir = find_root_dir()

# add the root directory to the system path
sys.path.append(root_dir)

# import custom modules
from src import data

## Load Datasets
The custom ``data`` module contains a function to load all datasets from ``.csv`` files from within a given directory.
The datasets will be loaded as dataframes (pandas or polars), and stored within one single dictionary:

In [ ]:
data_directory_uniform = '../../../data/datasets/01_uniform'

datasets = data.dict_from_directory(
    directory=data_directory_uniform,
    type='polars'
)

You can access the dataframes within this and all other dictionaries throught the project by their respective keys:

In [3]:
print(list(datasets.keys()))

['adhd', 'animal_depression', 'atypical_antipsychotics', 'calcium_channel_blockers', 'oral_hypoglycemics']


For example:

In [4]:
datasets['animal_depression'].head()

include,title,abstract,doi,literature_id,openalex_id
bool,str,str,str,i64,str
false,null,null,"""https://doi.org/10.1042/bj1300…",4656804,"""https://openalex.org/W24010252…"
false,null,null,null,6542443,"""https://openalex.org/W24105122…"
false,null,null,null,null,"""https://openalex.org/W24180790…"
true,null,null,"""https://doi.org/10.1111/ejn.12…",24188077,"""https://openalex.org/W20173882…"
false,null,null,"""https://doi.org/10.1097/000032…",11395604,"""https://openalex.org/W19957205…"


## Query API
Next we will programmatically query the OpenAlex API using each article's OpenAlex identifier to retrieve its title and abstract.

> **NOTE**
>
> Providing your E-Mail address in the cell below will give you access to the [polite pool](https://docs.openalex.org/how-to-use-the-api/rate-limits-and-authentication#the-polite-pool) with more consistent response times:

In [ ]:
import pyalex

YOUR_EMAIL = ''

if not YOUR_EMAIL == '':
    pyalex.config.email = YOUR_EMAIL

In [ ]:
import pyalex, polars as pl
from pyalex import Work
from tqdm.notebook import tqdm

# Process each dataset in the datasets dictionary
for dataset_name, data in tqdm(datasets.items(), desc='Downloading datasets'):

    # add titles & abstracts as whole columns later
    titles = []
    abstracts = []

    # Iterate through rows using Polars methods
    for row in tqdm(data.iter_rows(named=True), total=data.height, desc=dataset_name):

        article_id = row['openalex_id']

        # Check if openalex_id is null/missing
        if article_id is None:
            titles.append(None)
            abstracts.append(None)
        else:
            try:
                # retrieve title/abstract through the api
                openalex_data = pyalex.Works()[article_id]

                if isinstance(openalex_data, Work):
                    # Append title and abstract to the lists
                    titles.append(openalex_data['title'])
                    abstracts.append(openalex_data['abstract'])
                else:
                    # If the API call does not return a Work object, append None
                    print(f"API call did not return a Work object for {article_id}.")
                    titles.append(None)
                    abstracts.append(None)
            except Exception as e:
                # Handle API errors gracefully
                print(f"Error fetching data for {article_id}: {e}")
                titles.append(None)
                abstracts.append(None)

    # Add new columns to the Polars dataframe
    data = data.with_columns([
        pl.Series("title", titles),
        pl.Series("abstract", abstracts)
    ])

    # Update the datasets dictionary with the modified dataframe
    datasets[dataset_name] = data

adhd:   0%|          | 0/2 [00:00<?, ?it/s]

animal_depression:   0%|          | 0/2 [00:00<?, ?it/s]

atypical_antipsychotics:   0%|          | 0/2 [00:00<?, ?it/s]

calcium_channel_blockers:   0%|          | 0/2 [00:00<?, ?it/s]

oral_hypoglycemics:   0%|          | 0/2 [00:00<?, ?it/s]

## Save locally
Save the downloaded data locally:

In [ ]:
directory_to_save = '../../../data/datasets/02_openalex'

if os.path.exists(directory_to_save):
    for subject, dataframe in datasets.items():
        dataframe.write_csv(f'{directory_to_save}/{subject}_openalex.csv')
    print("Successfully saved all datasets.")
else:
    raise(FileNotFoundError(f"Directory {directory_to_save} does not exist. Please create it before saving the datasets."))